# **Hybrid Collaborative Filtering Book Recommender based on Popularity, KNN, SVD**

## **Load saved pkls & csvs**

In [3]:
import joblib
import pandas as pd

dir_knn = "knn_pkls"
dir_svd = "svd_pkls"
dir_pop = "popularity_csv"



# Load SVD
svd_model = joblib.load(f"{dir_svd}/svd_model.pkl")
train_ratings_svd = pd.read_csv(f"{dir_svd}/train_ratings_svd.csv")
test_ratings = pd.read_csv(f"{dir_svd}/test_ratings_svd.csv")

# Load KNN
knn_model = joblib.load(f"{dir_knn}/knn_model.pkl")
train_ratings_knn = pd.read_csv(f"{dir_knn}/train_ratings_knn.csv")
book_pivot_knn = joblib.load(f"{dir_knn}/book_pivot_knn.joblib")

# Load popularity
popularity_df = pd.read_csv(f"{dir_pop}/popularity_df.csv")

In [4]:
train_ratings_svd.head(3)

,user_id,isbn,ratings,title,author,year,publisher,img_url
0,254,0451150244,0,Pet Sematary,Stephen King,1984,Signet Book,http://images.amazon.com/images/P/0451150244.0...
1,254,0441003257,0,Good Omens,Neil Gaiman,1996,Ace Books,http://images.amazon.com/images/P/0441003257.0...
2,254,0316569321,0,White Oleander : A Novel,Janet Fitch,1999,"Little, Brown",http://images.amazon.com/images/P/0316569321.0...


In [5]:
test_ratings.head(3)

,user_id,isbn,ratings,title,author,year,publisher,img_url
0,254,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,http://images.amazon.com/images/P/0060930535.0...
1,254,0141301066,0,Matilda,Roald Dahl,1998,Puffin Books,http://images.amazon.com/images/P/0141301066.0...
2,254,0446605239,0,The Notebook,Nicholas Sparks,1998,Warner Books,http://images.amazon.com/images/P/0446605239.0...


In [6]:
train_ratings_knn.head(3)

,user_id,isbn,ratings,title,author,year,publisher,img_url
0,254,0439136350,9,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,1999,Scholastic,http://images.amazon.com/images/P/0439136350.0...
1,254,014028009X,0,Bridget Jones's Diary,Helen Fielding,1999,Penguin Books,http://images.amazon.com/images/P/014028009X.0...
2,254,067976402X,0,Snow Falling on Cedars,David Guterson,1995,Vintage Books USA,http://images.amazon.com/images/P/067976402X.0...


In [7]:
popularity_df.head(3)

,title,rating_count,avg_rating,score
0,The Lovely Bones: A Novel,225,3.355556,755.0
1,Harry Potter and the Chamber of Secrets (Book 2),141,3.978723,561.0
2,The Da Vinci Code,182,2.901099,528.0


In [8]:
book_pivot_knn

user_id,254,2276,2766,2977,3363,4017,6242,6251,6323,6543,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Is for Alibi (Kinsey Millhone Mysteries (Paperback)),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Map of the World,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
White Oleander : A Novel,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
White Oleander : A Novel (Oprah's Book Club),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Wicked: The Life and Times of the Wicked Witch of the West,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## **normalize popularity scores**

In [9]:
def normalizer_for_pop(popularity_df):

    # Crate new column for popularity score based on weights
    if "pop_score" not in popularity_df.columns:
        # Weighted combination of avg_rating and rating_count
        popularity_df["pop_score"] = (
            popularity_df["avg_rating"] * 0.7 + popularity_df["rating_count"] * 0.3
        )
    # Normalize to 0-1
    popularity_df["pop_score"] /= popularity_df["pop_score"].max()

    return popularity_df

In [10]:
normalizer_for_pop(popularity_df)
popularity_df.head()

,title,rating_count,avg_rating,score,pop_score
0,The Lovely Bones: A Novel,225,3.355556,755.0,0.760838
1,Harry Potter and the Chamber of Secrets (Book 2),141,3.978723,561.0,0.491095
2,The Da Vinci Code,182,2.901099,528.0,0.616858
3,To Kill a Mockingbird,138,3.644928,503.0,0.478747
4,Harry Potter and the Prisoner of Azkaban (Book 3),109,4.394495,479.0,0.389696


## **predict SVD scores for a user**

In [11]:
# Get the svd predicted scores(ratings) for unseen books i.e) boooks that user hasn't rated for a particular user

"""
scores finally should look :

"title" : normalized ratings of model predicted ratings

scores = {
 'Cradle and All': 0.0,             # Score = 0 means : Model thinks the user probably won’t like these
 'Disclosure': 0.19939775974208668,
 'Southern Cross': 0.3618182812111125,
 'Watership Down': 0.0,
 'When the Wind Blows': 0.0,
 "The Handmaid's Tale": 0.48606019878318346,
 'Skipping Christmas': 0.8556410644843749,...
 }
"""

def svd_scores_for_user(user_id, train_ratings_svd):
    all_books = train_ratings_svd["title"].unique()
    rated_books = train_ratings_svd[train_ratings_svd["user_id"]==user_id]["title"]
    unseen_books = [b for b in all_books if b not in rated_books.values]
    
    scores = {}
    for book in unseen_books:
        scores[book] = svd_model.predict(user_id, book).est
    
    # Normalize
    if scores:
        max_score = max(scores.values())
        for book in scores:
            scores[book] /= max_score
            
    # return list of books that can be suggest to a particular user with model predeicted score that measure how far each book related
    return scores # dictionary of (title : normalized_score of predicted ratings) 

In [26]:
dic_svd_score = svd_scores_for_user(254, train_ratings_svd) # based on user
dic_svd_score # all score values for books for user 254 / how much each book match with user 0 - not, 1 - best

{'Midnight in the Garden of Good and Evil: A Savannah Story': 0.42347541265781424,
 'The Bourne Ultimatum': 0.22302058594815127,
 'The King of Torts': 0.2888509563085016,
 'The Eight': 0.285132655344653,
 'Skipping Christmas': 0.678461317691598,
 '2nd Chance': 0.43235905557280147,
 'Contagion': 0.1933505691835462,
 'Roses Are Red (Alex Cross Novels)': 0.39330010907768953,
 'The Brethren': 0.41289711225241904,
 'The Valley of Horses': 0.5167359960331201,
 'The Tale of the Body Thief (Vampire Chronicles (Paperback))': 0.43666123380205707,
 'Disclosure': 0.2993349528778847,
 'Dust to Dust': 0.22828950014272842,
 'Watership Down': 0.5407455932634624,
 'Insomnia': 0.509720380759488,
 'Touching Evil': 0.3478383835141912,
 'Misery': 0.44960200504707715,
 'Nickel and Dimed: On (Not) Getting By in America': 0.6780920168416762,
 'The Dark Half': 0.34960052155893445,
 'Southern Cross': 0.4187350925324053,
 'The Runaway Jury': 0.5449202292500617,
 'When the Wind Blows': 0.4498788282886045,
 'Cause

## **get KNN scores for a user**

In [31]:
# get user id and how many suggesions we want
# then we find what are the books user aleready liked
# for each book user liked, we find n_neighbors no of similar books / (distance, indices) paires
# and also we will calculate scores, for first book we will add all similraty scores to scores dic
# but in next iteration of liked_books list we willupdate exsing ones or add new if not exist fromeralier  loops
# finaly we may have sevaral times updated dic of  books for a particular user
def knn_scores_for_user(user_id, train_ratings_knn, pivot, n_neighbors=20):

    # books user rated highly (liked books)
    liked_books = train_ratings_knn[
        (train_ratings_knn["user_id"] == user_id) & (train_ratings_knn["ratings"] >=8)
        ]["title"].values

    scores = {}

    for book in liked_books:
        
        # skip if book not in pivot
        if book not in pivot.index: # pivot.index is title
            continue

        # get similar books
        # asks the trained KNN model: "Find books that are most similar to this book" (like wise go thorugh all book in liked_books)
        # pivot.loc[book] > selecting one row by book name, eg: Harry Potter then  [5, 0, 4, ...] > rating fingerprint of the book
        # KNN expects 2D input (matrix form) so reshape it > [[5, 0, 4, 5]]
        # trained model looks at all books it learned during .fit() and asks : Which books have rating patterns most similar to this vector?
        # Since used cosine similarity, KNN compares angles between vectors
        # returns >>> distances = [[0.00, 0.21, 0.33, 0.41, 0.55, ...]], indices = [[10, 25, 3, 17, 8, ...]] 20 suggestions for 'book'
        distance, indices = knn_model.kneighbors(
            pivot.loc[book].values.reshape(1, -1),
            n_neighbors=n_neighbors
        )

        # convert distance > similarity
        # Distance meaning: 0 = identical, 1 = totally different
        # But recommendation logic wants:1 = identical, 0 = no similarity
        # So flip it by 1 - ....
        similarities = 1 - distance.flatten() # flatten to 1D [0.00, 0.21, 0.33, 0.41, 0.55]

        # Converts positions >  book names
        # pivot.index[25] → "LOTR"
        # so let put 1D indices array into oit and get titles 1D array
        neighbors = pivot.index[indices.flatten()]

        # finally
        # neighbors    > book names
        # similarities > similarity score for each book

        # using above let create score dic
        # neighbors    = ["HP", "LOTR", "Twilight", "Narnia"]
        # similarities = [1.00, 0.79, 0.67, 0.59]
        # zip(neighbors, similarities) pairs them as ("HP", 1.00), loop runs once per neighbor
        for neighbor, similarity in zip(neighbors, similarities): # n_neighbors time this will loops

            # skip current book
            if neighbor == book:
                continue
                
            # If book already has score → get it, otherwise start from 0 > 0 + 0.67
            # then why (neighbor, 0) bcs, in next iteration > Suppose another liked book also suggests this book, scores["book1"] = 0.67 + 0.52 = 1.19
            scores[neighbor] = scores.get(neighbor, 0) + similarity
            
    # normalize 0–1
    if scores:
        max_score = max(scores.values())

        # Here, book is the key in each key-value pair, normaly loop using key
        for book in scores:
            scores[book] /= max_score

    # Which other books have rating patterns across users most similar to this liked book?
    # we have loop through all books user liked so finally we may have score dic that with several times updated same book to get strong or only one
    # then we may have more thn 20- books in score, bcs liked book list may add new ones
    return scores # title : distance from considered book(cumulative sum)

In [25]:
dic_knn_score = knn_scores_for_user(254, train_ratings_knn, book_pivot_knn, 20) # based on book
dic_knn_score # all score values for books for user 254 / how much each book match with user 0 - not, 1 - best

{'Harry Potter and the Goblet of Fire (Book 4)': 1.0,
 'Harry Potter and the Chamber of Secrets (Book 2)': 0.9950549664448084,
 'Harry Potter and the Order of the Phoenix (Book 5)': 0.904902662356348,
 "Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))": 0.5711971345239413,
 'The Fellowship of the Ring (The Lord of the Rings, Part 1)': 0.46502403324394476,
 'A Time to Kill': 0.33714300274809084,
 'The Joy Luck Club': 0.294357340955548,
 'Artemis Fowl (Artemis Fowl, Book 1)': 0.10667833698199361,
 'Interview with the Vampire': 0.19514078261104056,
 'The Firm': 0.259452510454476,
 "The Bonesetter's Daughter": 0.40308925481594876,
 'The Alienist': 0.3556234250333667,
 'Hannibal': 0.08727236826552717,
 "Tuesdays with Morrie: An Old Man, a Young Man, and Life's Greatest Lesson": 0.08713181800251438,
 'Red Dragon': 0.1945652721203487,
 "Full House (Janet Evanovich's Full Series)": 0.0777558157033673,
 'The Testament': 0.1567882585020533,
 'Wicked: The Life and Times of the Wic

## Note

- SVD: I think the user would give this exact rating to an unseen book
- KNN: This book is similar to books the user liked. If they liked the originals, they will probably like this too, it will return how far each book from considering book as distance

## **Hybrid recommendation function**

## Why only unseen books

- Recommendations are meant to help discover new items the user hasn’t interacted with
- A user already knows or rated these books → no point recommending again
- so we only recommend unseen books

In [29]:
"""
    Hybrid recommendation combining popularity, KNN, and SVD scores.

    Parameters:
    - user_id: ID of the user
    - train_ratings_svd: DataFrame of all user ratings
    - svd_scores: dict of SVD predicted scores for the user
    - knn_scores: dict of KNN similarity scores for the user
    - top_n: number of recommendations to return
    - weights: dictionary of weights for pop/knn/svd

    Returns:
    - List of tuples: [(book_title, hybrid_score), ...] sorted descending
    """

def hybrid_recommend(user_id, n_suggestions=5, weights={"pop":0.2, "knn":0.5, "svd":0.3}):
    
    # Get unseen books
    all_books = train_ratings_svd["title"].unique() # unique titles not duplicated includes
    rated_books = train_ratings_svd[train_ratings_svd["user_id"]==user_id]["title"] # only book titles for particular user (user rated books)
    unseen_books = [b for b in all_books if b not in rated_books.values] # remove user already rated books 

    knn_scores = knn_scores_for_user(user_id, train_ratings_knn, book_pivot_knn, 20) # Calculated based on Book
    svd_scores = svd_scores_for_user(user_id, train_ratings_svd) # Calculated based on user

    
    hybrid_scores = {}

    for book in unseen_books:
        # Creates a boolean mask where only rows with book_title equal to your book variable are True
        # Select the pop_score column only for the rows where the mask is True
        # .values = Converts the resulting pandas Series into a NumPy array
        pop = popularity_df.loc[popularity_df["title"] == book, "pop_score"].values # since returning series we have always use 0 to access elt
        pop = pop[0] if len(pop) > 0 else 0

        # Get KNN and SVD scores (fallback to 0 if not found)
        knn = knn_scores.get(book, 0)
        svd = svd_scores.get(book, 0)

        # Now we have 3 metrices which is calculated accrofing to the Users, Books, Most rated, Let combine all together to get new book recoomendatoins,
        # Keep in mind, these all three mtrics have values  for each book how much is it fit to 
        # Weighted sum, hybrid_scores[book] default refer to key in score dic
        hybrid_scores[book] = weights["pop"] * pop + weights["knn"] * knn + weights["svd"] * svd

    # Sorts the keys (book titles) by their score in descending order
    # .items() returns list of key-value pairs as tuples (key, value)
    # [("Harry Potter 1", 0.78), ("LOTR1", 0.40), ...]
    # key=lambda x: x[1] → sort by the second element of each tuple (the score), x is a tuple (book_title, score), x[1] = score, descending/
    top_n_books = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n_suggestions]
    
    return top_n_books

## Testing variabels

In [23]:
# # train_ratings_svd["title"].unique()
# ''' all_books
# array(['Pet Sematary', 'Good Omens', 'White Oleander : A Novel',
#        'The Subtle Knife (His Dark Materials, Book 2)',
#        'The Secret Life of Bees', 'Chocolat', 'Rising Tides',
#        'American Gods', 'Circle of Friends',
#        "She's Come Undone (Oprah's Book Club)", 'Night Whispers',
# '''

# # train_ratings_svd[train_ratings_svd["user_id"]==254]["title"]
# '''rated_books
# 0                                          Pet Sematary
# 1                                            Good Omens
# 2                              White Oleander : A Novel
# 3         The Subtle Knife (His Dark Materials, Book 2)
# 4                               The Secret Life of Bees
# 5                                              Chocolat
# 6                                          Rising Tides
# .
# .
# .
# '''

# # a = train_ratings_svd["title"].unique()
# # r =  train_ratings_svd[train_ratings_svd["user_id"]==254]["title"]
# # [b for b in a if b not in r.values]

# '''unseen_books
# ['Midnight in the Garden of Good and Evil: A Savannah Story',
#  'The Bourne Ultimatum',
#  'The King of Torts',
#  'The Eight',
#  'Skipping Christmas',....
#  ]
#  '''

# pop = popularity_df.loc[popularity_df["title"] == 254, "pop_score"].values
# '''
# array([0.08137981])
# '''
# pop

# pop = pop[0] if len(pop) > 0 else 1000
# pop

## **Predictions**

In [24]:
user_id_example = test_ratings["user_id"].iloc[50]
n_suggestions =10

rec_books = hybrid_recommend(
    user_id=user_id_example,
    n_suggestions=n_suggestions
)

print(f"Top {n_suggestions} Hybrid Recommendations for user {user_id_example}:\n")

for i in rec_books:
    print(i)

Top 10 Hybrid Recommendations for user 4017:

('Empire Falls', 0.7556700721100944)
('House of Sand and Fog', 0.6424764380544807)
('A Map of the World', 0.6311329760847699)
("Bridget Jones's Diary", 0.6109229190225414)
('The Da Vinci Code', 0.6097029070028901)
('The Poisonwood Bible: A Novel', 0.6009981510966321)
('SHIPPING NEWS', 0.5612083491139638)
('Wicked: The Life and Times of the Wicked Witch of the West', 0.533728212267795)
("The Pilot's Wife : A Novel", 0.5334453108086668)
('Fried Green Tomatoes at the Whistle Stop Cafe', 0.5294668206157682)


## **Evaluate hybrid model**

In [16]:
def precision_at_n_single_user(user_id, test_ratings_user, N=5, rate_threshold=8, top_n=10):
    '''
    recommended_books: list of book titles we got as recommendation as top_n_books_by_hybrid
    test_ratings_user: DataFrame of user's test ratings
    rate_threshold: rating to consider as user liked
    '''
    
    liked_books = test_ratings_user[
        (test_ratings_user["user_id"]==user_id) & (test_ratings_user["ratings"] >= rate_threshold)
        ]["title"].tolist()
    '''
    array(['The Dark Half', "Harry Potter and the Sorcerer's Stone (Book 1)"],
      dtype=object)
    '''
    # print(liked_books, "\n\n")
    if len(liked_books) == 0:
        return 0  # cannot evaluate

    # Getting recommendations
    recs = hybrid_recommend(
        user_id=user_id,
        train_ratings_svd= train_ratings_svd,
        train_ratings_knn=train_ratings_knn,
        book_pivot_knn=book_pivot_knn,
        top_n=10
    )

    if top_n < N:
        print("Values exceeded top_n < N")
        N = top_n
    
    # recommended_books[:N] >>> slices your recommended_books list to only take the top N recommendations
    # iterates over each book in the top N recommended books
    # if book in liked_books checks whether the current recommended book is in the list of books the user actually liked
    # If the condition is True, it adds 1 to the list. If False, it adds nothing
    recs = recs[:N] # when evaluating consider sevaral items bcs sometimews 10 is not good

    hits = []

    for book, score in recs:
        # print("book in recommended_books : ", book)
        if book in liked_books:
            hits.append(1)
            
    # print("hits : ", hits)
    precision = sum(hits) / N

    return precision

In [17]:
# Testing

liked_books = test_ratings[
        (test_ratings["user_id"] == 254) & (test_ratings["ratings"] >= 8)
        ]["title"].values

liked_books

array(['The Dark Half', "Harry Potter and the Sorcerer's Stone (Book 1)"],
      dtype=object)

In [18]:
precision = precision_at_n_single_user(
    user_id = user_id_example, 
    test_ratings_user = test_ratings, 
    N = 5,
    rate_threshold = 8,
    top_n = 10
)
print(f"precisison for user_id {user_id_example} : {precision*100} %")

precisison for user_id 4017 : 20.0 %


In [19]:
import numpy as np

precisions = []
count=0

for user_id, user_group  in test_ratings.groupby("user_id"):
    
    count += 1
    
    precision = precision_at_n_single_user(
        user_id = user_id, 
        test_ratings_user = user_group, # ONLY this user's test data
        N=5,
        rate_threshold=8, 
        top_n=10
    )

    # skip users with no liked books
    if precision is not None:
        precisions.append(precision)
    
    precisions.append(precision)
    
    if count % 100 == 0:
        print(count, " users cheked, so far precisison mean : ", np.mean(precisions))
        
print("\nFinal Precision@5:", np.mean(precisions))

100  users cheked, so far precisison mean :  0.07
200  users cheked, so far precisison mean :  0.07200000000000001
300  users cheked, so far precisison mean :  0.06866666666666667
400  users cheked, so far precisison mean :  0.0635
500  users cheked, so far precisison mean :  0.0608
600  users cheked, so far precisison mean :  0.05533333333333334
700  users cheked, so far precisison mean :  0.05542857142857143
800  users cheked, so far precisison mean :  0.0565

Final Precision@5: 0.055503512880562066


## Model Comparison — Precision@N

#### Popularity  = 0.011441307578008916
#### KNN         = 0.016956521739130436
#### SVD         = 0.021690590111642746
#### Hubrid      = 0.055503512880562066